In [ ]:
from collections import defaultdict

import json
import numpy as np
import pandas as pd
import pickle
import polars as pl

import torch
from sentence_transformers import SentenceTransformer

from tqdm import tqdm as tqdm


In [ ]:
interactions_dataset_path = '../data/Beauty/Beauty_5.json'
metadata_path = '../data/Beauty/metadata.json'

interactions_output_path = '../data/Beauty/inter.json'
embeddings_output_path = '../data/Beauty/content_embeddings.pkl'


In [ ]:
df = defaultdict(list)

with open(interactions_dataset_path, 'r') as f:
    for line in f.readlines():
        review = json.loads(line)
        df['user_id'].append(review['reviewerID'])
        df['item_id'].append(review['asin'])
        df['timestamp'].append(review['unixReviewTime'])

print(f'Number of events: {len(df["user_id"])}')

df = pl.from_dict(df).with_row_index('_row_idx')


In [ ]:
df.head()

In [ ]:
filtered_df = df.clone()

In [ ]:
# Processing dataset to get core-5 state in case full dataset is provided
is_changed = True
threshold = 5
good_users = set()
good_items = set()

while is_changed:
    user_counts = filtered_df.group_by('user_id').agg(
        pl.len().alias('user_count'),
    )
    item_counts = filtered_df.group_by('item_id').agg(
        pl.len().alias('item_count'),
    )

    good_users = user_counts.filter(pl.col('user_count') >= threshold).select(
        'user_id',
    )
    good_items = item_counts.filter(pl.col('item_count') >= threshold).select(
        'item_id',
    )

    old_size = len(filtered_df)
    filtered_df = filtered_df.join(
        good_users, on='user_id', how='inner',
    ).join(
        good_items, on='item_id', how='inner',
    )

    if len(filtered_df) == old_size:
        is_changed = False

filtered_df = filtered_df.sort('_row_idx').drop('_row_idx')


In [ ]:
unique_values = filtered_df["user_id"].unique(maintain_order=True).to_list()
user_ids_mapping = {value: i for i, value in enumerate(unique_values)}

filtered_df = filtered_df.with_columns(
    pl.col("user_id").replace_strict(user_ids_mapping)
)

unique_values = filtered_df["item_id"].unique(maintain_order=True).to_list()
item_ids_mapping = {value: i for i, value in enumerate(unique_values)}

filtered_df = filtered_df.with_columns(
    pl.col("item_id").replace_strict(item_ids_mapping)
)

filtered_df.head()

In [ ]:
item_ids_mapping_df = pl.from_dict({
    'old_item_id': list(item_ids_mapping.keys()),
    'new_item_id': list(item_ids_mapping.values())
})
item_ids_mapping_df.head()

In [ ]:
filtered_df.head()

In [ ]:
filtered_df = filtered_df.sort(["user_id", "timestamp"])

grouped_filtered_df = filtered_df.group_by("user_id", maintain_order=True).agg(pl.all())

In [ ]:
item_ids_mapping_df.head()

In [ ]:
grouped_filtered_df.head()

In [ ]:
print('Users count:', filtered_df.select('user_id').unique().shape[0])
print('Items count:', filtered_df.select('item_id').unique().shape[0])
print('Actions count:', filtered_df.shape[0])
print('Avg user history len:', np.mean(list(map(lambda x: x[0], grouped_filtered_df.select(pl.col('item_id').list.len()).rows()))))

In [ ]:
json_data = {}
for user_id, item_ids, _ in grouped_filtered_df.iter_rows():
    json_data[user_id] = item_ids

with open(interactions_output_path, 'w') as f:
    json.dump(json_data, f, indent=2)

## Content embedding creation

In [ ]:
def getDF(path):
    i = 0
    df = {}
    with open(path, 'r') as f:
        for line in f.readlines():
            df[i] = eval(line)
            i += 1

    return pd.DataFrame.from_dict(df, orient="index")

df = getDF(metadata_path)
df.head()

In [ ]:
def _safe(value, default: str = 'Unknown') -> str:
    """Handle missing/null values for metadata fields."""
    if value is None:
        return default
    if isinstance(value, float) and pd.isna(value):
        return default
    if isinstance(value, list) and not value:
        return default
    if isinstance(value, str) and not value.strip():
        return default
    return str(value)


def preprocess(row: pd.Series):
    title = _safe(row.get('title'))
    price = _safe(row.get('price'))
    brand = _safe(row.get('brand'))
    raw_categories = row.get('categories')
    if isinstance(raw_categories, list) and raw_categories:
        flat = raw_categories[0] if isinstance(raw_categories[0], list) else raw_categories
        categories = ', '.join(str(c) for c in flat) if flat else 'Unknown'
    else:
        categories = 'Unknown'
    return f'Title: {title} Price: {price} Brand: {brand} Categories: {categories}'


def get_data(metadata_df, item_ids_mapping_df):
    filtered_df = metadata_df.join(
        item_ids_mapping_df,
        left_on='asin',
        right_on='old_item_id',
        how='inner',
    ).select(
        pl.col('new_item_id'),
        pl.col('title'),
        pl.col('price'),
        pl.col('brand'),
        pl.col('categories'),
    )
    filtered_df = filtered_df.to_pandas()
    filtered_df['combined_text'] = filtered_df.apply(preprocess, axis=1)
    return filtered_df


In [ ]:
data = get_data(pl.from_pandas(df), item_ids_mapping_df)


In [ ]:
# Sentence-T5 for item content embeddings
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SentenceTransformer('sentence-transformers/sentence-t5-base', device=device)
model.eval()

texts = data['combined_text'].tolist()
item_ids = data['new_item_id'].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f'Embeddings shape: {embeddings.shape}')  #  (N, 768)

new_df = {
    'item_id': item_ids,
    'embedding': embeddings.tolist(),
}

with open(embeddings_output_path, 'wb') as f:
    pickle.dump(new_df, f)

print(f'Saved: {embeddings_output_path}')
